In [ ]:
import pandas as pd
import numpy as np

print("🚀 KAPSAMLI VERİ TEMİZLİĞİ (PIPELINE) BAŞLIYOR...\n")

# 1. VERİLERİ YÜKLE
# Kendi bilgisayarının gücüne göre nrows'u kaldırabilir veya artırabilirsin
df_yolculuk = pd.read_csv('../data/YOLCULUK_2025_H1.csv', nrows=500000) 
d_hat = pd.read_csv('../data/D_HAT.csv', encoding='iso-8859-9')

print(f"📊 İlk Durum: {len(df_yolculuk)} satır yolculuk verisi.")

# 2. EKSİK (NaN) VERİ KONTROLÜ
print("\n--- 1. ADIM: Eksik Veri Temizliği ---")
eksik_sayisi = df_yolculuk.isnull().sum()
print("Sütunlardaki Boşluklar:\n", eksik_sayisi[eksik_sayisi > 0])

# Zaman (Tarih/Saat) verisi boş olanları veya anlamsız olanları atıyoruz
if 'ISLEMZAMANI' in df_yolculuk.columns:
    df_yolculuk = df_yolculuk.dropna(subset=['ISLEMZAMANI'])

# 3. İLK BİNİŞ VE AKTARMA AYRIMI
print("\n--- 2. ADIM: Aktarma Mantıksal Filtresi ---")
# SIDONCEKIHAT 0 ise ilk biniştir. NaN olanları 0 yapıyoruz (ilk biniş kabul ediyoruz)
df_yolculuk['SIDONCEKIHAT'] = df_yolculuk['SIDONCEKIHAT'].fillna(0)

# Sadece "Gerçek Aktarmaları" ayıklıyoruz (Ağ algoritmamız için)
df_aktarma = df_yolculuk[df_yolculuk['SIDONCEKIHAT'] != 0].copy()
print(f"✅ Sadece gerçek aktarmalar filtrelendi: {len(df_aktarma)} satır kaldı.")

# 4. REFERANS BÜTÜNLÜĞÜ (Hayalet Hatları Temizleme)
print("\n--- 3. ADIM: D_HAT ile Eşleştirme Bütünlüğü ---")
gecerli_sidler = d_hat['SID'].unique()

# SIDHAT veya SIDONCEKIHAT'ı D_HAT tablosunda OLMAYAN satırları siliyoruz (Çöp veri)
oncesi_satir = len(df_aktarma)
df_aktarma = df_aktarma[df_aktarma['SIDHAT'].isin(gecerli_sidler)]
df_aktarma = df_aktarma[df_aktarma['SIDONCEKIHAT'].isin(gecerli_sidler)]
silinen_hayalet = oncesi_satir - len(df_aktarma)
print(f"👻 Sistemde kaydı olmayan {silinen_hayalet} adet 'hayalet' aktarma silindi.")

# 5. AYKIRI DEĞER (OUTLIER) TEMİZLİĞİ (Aynı hatta hemen aktarma yapılamaz)
print("\n--- 4. ADIM: Anomali Temizliği ---")
# Yolcu 11C'den inip saniyeler içinde tekrar 11C'ye binmiş gibi görünen 
# mükerrer basımları (cihaz hatalarını) temizliyoruz.
ayni_hat_hatasi = len(df_aktarma[df_aktarma['SIDHAT'] == df_aktarma['SIDONCEKIHAT']])
df_aktarma = df_aktarma[df_aktarma['SIDHAT'] != df_aktarma['SIDONCEKIHAT']]
print(f"🚫 Aynı hatta art arda basım hatası olan {ayni_hat_hatasi} satır temizlendi.")

# 6. TEMİZ VERİYİ DIŞARI AKTAR (Export)
temiz_dosya_yolu = '../data/TEMIZ_AKTARMA_VERISI.csv'
df_aktarma.to_csv(temiz_dosya_yolu, index=False)
print(f"\n🎉 TEMİZLİK BİTTİ! Yapay Zeka için hazır temiz veri kaydedildi: {temiz_dosya_yolu}")
print(f"Son Kullanılabilir Veri Boyutu: {len(df_aktarma)} satır.")